In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import qr
from scipy.optimize import linprog, minimize
import scipy.sparse as sp
from scipy.special import logsumexp, softmax
from time import perf_counter


In [ ]:

def pinwheel(a, b, width=0.3, wobble=0.15, nb=48, ni=40, seed=0):
    'Three-class pinwheel: a dataset of shape (264, 2) and its labels in range(3).'
    rng = np.random.default_rng(seed)
    T = np.linspace(0, 2*np.pi, nb, endpoint=False)
    U = rng.uniform(size=(ni, 2))
    cx, L = (a + b)/2, (b - a)/2
    r = 1 + wobble*(0.5*np.sin(3*T) + 0.3*np.cos(5*T + 1))
    bd = np.stack([cx + L*r*np.cos(T), 1 + width*r*np.sin(T)], 1)
    rr, th = np.sqrt(U[:, 0])*0.9, 2*np.pi*U[:, 1]
    rin = 1 + wobble*(0.5*np.sin(3*th) + 0.3*np.cos(5*th + 1))
    inn = np.stack([cx + L*rr*rin*np.cos(th), 1 + width*rr*rin*np.sin(th)], 1)
    base = np.vstack([bd, inn])
    rot = lambda t: np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
    X = np.vstack([base @ rot(2*np.pi*i/3).T for i in range(3)])
    labels = np.repeat(np.arange(3), len(base))
    return X, labels


In [ ]:

def reduce(X):
    'Features S such that X[:, S] is full-rank and x -> x[S] is one-to-one on the affine span of X.'
    Xc = X - X.mean(0)
    _, R, piv = qr(Xc, mode='economic', pivoting=True)
    tol = abs(R[0, 0])*max(Xc.shape)*np.finfo(float).eps
    return np.sort(piv[:int((abs(np.diag(R)) > tol).sum())])

def margins(Z, y, d):
    'Sparse G with (G @ W.ravel())[row] = y_i - y_j at x, for x in class i and j != i.'
    Z = sp.coo_matrix(Z)
    N, p = Z.shape
    rows, cols, vals, top = [], [], [], 0
    for j in range(d):
        idx = np.flatnonzero(y != j)
        pos = np.full(N, -1)
        pos[idx] = np.arange(len(idx))
        k = pos[Z.row] >= 0
        r, c, v = pos[Z.row[k]] + top, Z.col[k], Z.data[k]
        rows += [r, r]
        cols += [y[Z.row[k]]*p + c, j*p + c]
        vals += [v, -v]
        top += len(idx)
    return sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(top, d*p))

def feasible(G, strict, A_eq=None):
    'A solution w of G w >= 1 (strict), or of G w >= 0 with sum(G w) = 1, and A_eq w = 0; or None.'
    p = G.shape[1]
    if strict:
        b_ub, eq = -np.ones(G.shape[0]), A_eq
    else:
        b_ub = np.zeros(G.shape[0])
        eq = sp.csr_matrix(G.sum(0)) if A_eq is None else sp.vstack([sp.csr_matrix(G.sum(0)), A_eq])
    b_eq = None
    if eq is not None:
        b_eq = np.zeros(eq.shape[0])
        if not strict:
            b_eq[0] = 1
    res = linprog(np.zeros(p), A_ub=-G, b_ub=b_ub, A_eq=eq, b_eq=b_eq,
                  bounds=[(None, None)]*p, method='highs')
    return res.x if res.status == 0 else None

def separability(dataset, labels, sample=None, pairs=False, seed=0):
    '''Separability report for a multi-class dataset (N samples, n features) with labels in range(d).
    With sample = m, the linear programs use m random samples: 'not separable' results then hold for the whole dataset, while 'separable' results are inconclusive.'''
    t0 = perf_counter()
    X, y = np.asarray(dataset, float), np.asarray(labels)
    N, n = X.shape
    d = int(y.max()) + 1
    S = reduce(X)
    full = len(S) == n
    if sample is not None and sample < N:
        _, _, piv = qr(np.hstack([X[:, S], np.ones((N, 1))]).T, mode='economic', pivoting=True)
        base = piv[:len(S) + 1]
        rest = np.setdiff1d(np.arange(N), base)
        extra = np.random.default_rng(seed).choice(rest, max(sample - len(base), 0), replace=False)
        keep = np.r_[base, extra]
        Xs, ys = X[keep], y[keep]
    else:
        Xs, ys = X, y
    Z = np.hstack([Xs[:, S], np.ones((len(Xs), 1))])
    p = Z.shape[1]
    G = margins(Z, ys, d)
    cen = sp.hstack([sp.identity(p)]*d, format='csr')
    w = feasible(G, False, cen)
    W = None
    if w is not None:
        W = np.zeros((d, n + 1))
        W[:, np.r_[S, n]] = w.reshape(d, p)
    strict = feasible(G, True, cen) is not None
    off = []
    for i in range(d):
        s = np.where(ys == i, 1.0, -1.0)
        if feasible(sp.csr_matrix(s[:, None]*Z), False) is not None:
            off.append(i)
    info = dict(N=N, n=n, d=d, rank=len(S), full_rank=full, separable=w is not None,
                margin_separable=strict, splits_off=off, W=W)
    if pairs:
        info['pairwise'] = {(i, j): feasible(sp.csr_matrix(np.vstack([ Z[ys == i], -Z[ys == j]])), True) is not None for i in range(d) for j in range(i + 1, d)}
    tag = '' if sample is None or sample >= N else f' (LPs on {len(ys)} samples)'
    print(f'N = {N}, sample space = R^{n}, d = {d}{tag}')
    if not full:
        print(f'not full-rank: the affine span has dimension {len(S)}, so the dataset is trivially separable;'
              f' the tests below use the {len(S)} reduced features')
    word = (lambda b: 'yes' if b else 'no') if tag == '' else (lambda b: 'inconclusive' if b else 'no')
    print(f'separable: {word(info['separable'])},  margin separable: {word(strict)}')
    print(f'classes that split off: {off if off else 'none'}' + ('' if tag == '' else ' (from the sample)'))
    if pairs:
        bad = [k for k, v in info['pairwise'].items() if not v]
        print('pairwise margin separable: ' + ('all pairs' if not bad else f'all pairs except {bad}'))
    print(f'{perf_counter() - t0:.1f} s')
    return info


In [ ]:

def lift(P):
    'Augment samples: x -> (x, 1).'
    return np.hstack([P, np.ones((len(P), 1))])

def pair_line(A, B):
    'Maximum-margin line w = (m, b), with m.x + b >= 1 on A and <= -1 on B, or None if none exists.'
    Z = np.vstack([lift(A), -lift(B)])
    lp = linprog(np.zeros(3), A_ub=-Z, b_ub=-np.ones(len(Z)), bounds=[(None, None)]*3, method='highs')
    if lp.status != 0:
        return None
    res = minimize(lambda w: w[0]**2 + w[1]**2, lp.x, jac=lambda w: np.array([2*w[0], 2*w[1], 0.0]), constraints=[{'type': 'ineq', 'fun': lambda w: Z @ w - 1, 'jac': lambda w: Z}], method='SLSQP')
    return res.x

def gaps(W, X, y):
    'For each sample, its own level minus the highest other level.'
    Y = X @ W.T
    rows = np.arange(len(y))
    own = Y[rows, y].copy()
    Y[rows, y] = -np.inf
    return own - Y.max(1)

def mismatched(W, X, y):
    'Boolean array: which samples have y_i <= y_j for some j != i.'
    return gaps(W, X, y) <= 0


In [ ]:

def gd(X, y, iters=200_000):
    Pm = np.eye(3)[y]	# one-hot targets
    N = len(X)
    t = 1/np.mean(np.sum(X**2, 1))
    W = np.zeros((3, 3))
    norms, losses = [], []
    sep = None		# (n, W_n, seconds) at first separation
    rec = set(np.unique(np.logspace(0, np.log10(iters), 300).astype(int)))
    t0 = perf_counter()
    for n in range(1, iters + 1):
        Q = softmax(X @ W.T, axis=1)
        W -= t*(Q - Pm).T @ X/N
        if sep is None and not mismatched(W, X, y).any():
            sep = (n, W.copy(), perf_counter() - t0)
        if n in rec:
            Y = X @ W.T
            norms.append((n, np.linalg.norm(W)))
            losses.append((n, np.mean(logsumexp(Y, axis=1) - np.sum(Pm*Y, 1))))
    return W, np.array(norms), np.array(losses), sep, perf_counter() - t0


In [ ]:


PARAMS = [(-2, 0.3), (-2.5, 1.1), (-2.5, 1.3)]
runs = {}
for a, b in PARAMS:
    X, labels = pinwheel(a, b)
    print(f'\npinwheel (a,b) = ({a},{b})')
    info = separability(X, labels, pairs=True)
    Xa = lift(X)
    W, norms, losses, sep, seconds = gd(Xa, labels)
    grad = (softmax(Xa @ W.T, axis=1) - np.eye(3)[labels]).T @ Xa/len(Xa)
    C = [X[labels == i] for i in range(3)]
    runs[(a, b)] = dict(C=C, W=W, norms=norms, losses=losses, sep=sep, seconds=seconds, separable=info['separable'])
    print(f'   after {int(norms[-1, 0])} steps: |W| = {norms[-1, 1]:.2f}, loss = {losses[-1, 1]:.2e},  'f'|grad| = {np.linalg.norm(grad):.1e}, '  f'misclassified = {int(mismatched(W, Xa, labels).sum())} of {len(X)}, {seconds:.1f} s')
    print(f'   first separation: ' + (f'n = {sep[0]}' if sep else 'never'))



In [ ]:

plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif'], 'mathtext.fontset': 'stix', 'font.size': 8, 'axes.linewidth': 0.6, 'pdf.fonttype': 42})
cols = ['#c0392b', '#2e8b3a', '#2f5fb3']
marks = ['o', '^', 's']
tints = ['#f6dcd9', '#dcefdc', '#dce5f5']
LIM = 2.9
g = np.linspace(-LIM, LIM, 500)
X1, X2 = np.meshgrid(g, g)
GRID = np.stack([X1.ravel(), X2.ravel(), np.ones(X1.size)], 1)
BOX = [(-LIM, -LIM), (LIM, -LIM), (LIM, LIM), (-LIM, LIM)]

def clip(poly, a, c):
    'Clip the convex polygon poly to the half-plane a.x + c >= 0.'
    out = []
    for k in range(len(poly)):
        p, q = np.array(poly[k]), np.array(poly[(k + 1) % len(poly)])
        fp, fq = a @ p + c, a @ q + c
        if fp >= 0:
            out.append(p)
        if fp*fq < 0:		# the edge crosses the line
            out.append(p + fp/(fp - fq)*(q - p))
    return out

def region(W, i):
    'The region R_i of W inside the box, as a polygon; on ties the lower index wins.'
    poly = BOX
    for j in range(len(W)):
        if j != i:
            a, c = W[i, :2] - W[j, :2], W[i, 2] - W[j, 2]
            if np.allclose(a, 0) and np.isclose(c, 0) and j < i:
                return []		# identical rows
            poly = clip(poly, a, c)
            if not poly:
                return []
    return poly

def regions(ax, C, W, title):
    'Shade the regions of W and plot the three classes.'
    for i in range(len(W)):
        poly = region(W, i)
        if poly:
            ax.add_patch(plt.Polygon(poly, closed=True, facecolor=tints[i], edgecolor=tints[i], linewidth=0.3, zorder=0))
    for P, c, m in zip(C, cols, marks):
        ax.scatter(P[:, 0], P[:, 1], s=4, c=c, marker=m, linewidths=0)
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, 2.3); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=8)

fig, axs = plt.subplots(1, 3, figsize=(6.9, 2.5))
for ax, (a, b), lab in zip(axs, PARAMS, 'abc'):
    r = runs[(a, b)]
    if r['sep'] is not None:	# stop at first separation
        n_sep, W, _ = r['sep']
        sub = f'separable, stopped at $n={n_sep}$'
    else:
        W, sub = r['W'], 'not separable, $n=2\\times10^5$'
    regions(ax, r['C'], W, f'({lab}) $a={a},\\ b={b}$\n{sub}')
    if not r['separable']:		# pairwise separating lines
        for i, j in [(0, 1), (1, 2), (2, 0)]:
            w = pair_line(r['C'][i], r['C'][j])
            if w is not None:
                ax.contour(X1, X2, (GRID @ w).reshape(X1.shape), levels=[0], colors='k', linestyles='--', linewidths=0.7)
plt.tight_layout(pad=0.3, w_pad=0.8)
plt.savefig('pinwheel_gd.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()
